# ProductLens 01 · Training pipeline (Kaggle GPU)

End-to-end pipeline that produces every artifact ProductLens serves: data preparation, contrastive fine-tuning,
evaluation, catalog indexing and the browser export.

| Setting | Value |
|---|---|
| Accelerator | NVIDIA Tesla T4 (16 GB) |
| Internet | On (downloads ABO from its public S3 bucket) |
| Runtime | about 3.5 hours (5 epochs at about 39 minutes, plus indexing and export) |

The same pipeline can be submitted from the command line with
`python scripts/kaggle/submit_pipeline.py --user <kaggle-username>`.


## 1. Environment

In [ ]:
import os, subprocess
REPO = "https://github.com/ranjeet258/ProductLens.git"
WORK = "/kaggle/working"
if not os.path.exists("ProductLens"):
    subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
os.chdir("ProductLens")
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q open_clip_torch faiss-cpu onnx onnxruntime onnxscript

## 2. Data: Amazon Berkeley Objects

Two archives are needed: the listing metadata and the small (256 px) product images, about 3.5 GB in total.
They are extracted to `/tmp` so they do not count against the notebook's output quota.

In [ ]:
import glob
ABO = "/tmp/abo"
os.makedirs(ABO, exist_ok=True)
for name in ("abo-listings.tar", "abo-images-small.tar"):
    url = f"https://amazon-berkeley-objects.s3.amazonaws.com/archives/{name}"
    subprocess.run(f"cd {ABO} && wget -q {url} && tar xf {name} && rm {name}", shell=True, check=True)
img_meta = glob.glob(f"{ABO}/**/images.csv.gz", recursive=True)[0]
SMALL = os.path.join(os.path.dirname(os.path.dirname(img_meta)), "small")
print("image root:", SMALL)

## 3. Image-text pairs, split by product

English titles are combined with product type, colour, brand and one bullet point. The split is by `item_id`
(90 / 5 / 5), so no product appears in more than one split. Secondary photos are added as extra training pairs.

In [ ]:
!python scripts/prepare_abo.py --abo-root {ABO} --out-dir /tmp/data --use-other-images

## 4. Contrastive fine-tuning

OpenCLIP ViT-B/32 (LAION-2B), symmetric InfoNCE with a learnable temperature, AdamW (lr 1e-5, wd 0.1),
200 warmup steps then cosine decay, mixed precision, batch size 128. Zero-shot validation recall is recorded
before the first step, and the best epoch is kept.

In [ ]:
!python src/train.py --train-csv /tmp/data/train.csv --val-csv /tmp/data/val.csv --image-root {SMALL} \
    --epochs 5 --batch-size 128 --lr 1e-5 --workers 4 --out {WORK}/run

## 5. Held-out evaluation (zero-shot vs fine-tuned)

In [ ]:
!python src/evaluate.py --test-csv /tmp/data/test.csv --image-root {SMALL} \
    --checkpoint {WORK}/run/best.pt --workers 4 --out {WORK}/run/test_metrics.json
import json
print(json.dumps(json.load(open(f"{WORK}/run/test_metrics.json")), indent=2))

## 6. Catalog index

Every catalog photo (121,549 products) is embedded with the fine-tuned image encoder and stored in a FAISS
IVF-PQ index: 1,024 inverted lists, 32 sub-quantizers of 8 bits.

In [ ]:
!python src/build_index.py --catalog-csv /tmp/data/catalog.csv --image-root {SMALL} \
    --checkpoint {WORK}/run/best.pt --index-type ivfpq --workers 4 --out {WORK}/artifacts
import shutil
shutil.move(f"{WORK}/run/best.pt", f"{WORK}/artifacts/model.pt")
os.remove(f"{WORK}/run/last.pt")

## 7. Browser export (CPU)

Exports both encoders to ONNX, applies dynamic INT8 quantization with an accuracy gate (cosine > 0.97 against
PyTorch), and converts the index to flat binary arrays. This step can also run locally:
`python scripts/export_static.py --artifacts artifacts --out static_site --push <hf-username>`.

In [ ]:
!python scripts/export_static.py --artifacts {WORK}/artifacts --out {WORK}/static_site

## Outputs

| Path | Contents |
|---|---|
| `run/metrics.json` | zero-shot and per-epoch validation recall, loss, epoch time |
| `run/test_metrics.json` | held-out Recall@1/5/10, zero-shot vs fine-tuned |
| `artifacts/` | `model.pt`, `index.faiss`, `meta.parquet`, `index_info.json` (inputs to the API) |
| `static_site/` | INT8 ONNX encoders, binary index, product shards and the web page |

Copy `run/metrics.json` to `results/train_history.json`, `run/test_metrics.json` to `results/`, and the
`artifacts/` folder into the repository, then run `02_results_analysis.ipynb`.